# Workbook 05 — Timer & State-Machine Drills

**Sprint 5** — companion to Bolton Ch. 7–12.

## Tasks

1. Implement a TON in Python and trace its accumulator across an input pattern.
2. Build a 1 Hz flasher from two TONs.
3. Convert an analog 4–20 mA RTD reading to °C with linear scaling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.05  # 50 ms scan
duration = 10  # seconds
t = np.arange(0, duration, dt)

# Input pulse pattern: 2 s on, 1 s off, repeat
EN = ((t % 3) < 2).astype(int)

# TON: accumulator counts up while EN, resets when EN drops
PT = 1.5  # preset, seconds
ACC = np.zeros_like(t)
Q = np.zeros_like(t)
for i in range(1, len(t)):
    if EN[i]:
        ACC[i] = min(ACC[i-1] + dt, PT)
    else:
        ACC[i] = 0
    Q[i] = 1 if ACC[i] >= PT else 0

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(8, 4))
ax[0].step(t, EN, where='post'); ax[0].set_ylabel('EN')
ax[1].plot(t, ACC); ax[1].set_ylabel('ACC (s)')
ax[2].step(t, Q, where='post'); ax[2].set_ylabel('Q'); ax[2].set_xlabel('t (s)')
plt.tight_layout()

## Task 2 — 1 Hz flasher

Chain two TONs: T1 times 0.5 s with EN = NOT Q2, Q1 enables T2 which also times 0.5 s. Output toggles every 0.5 s.

Implement and plot.

## Task 3 — RTD scaling

An analog input card delivers 0–32 767 raw counts for 4–20 mA. The Pt100 transmitter is configured for 0–500 °C. Write a function that scales raw to °C and verify with a few sample points.

*Hint: at 4 mA → 0 raw → 0 °C; at 20 mA → 32 767 raw → 500 °C, linear in between.*

In [ ]:
def scale(raw, raw_min=0, raw_max=32767, eu_min=0.0, eu_max=500.0):
    return eu_min + (eu_max - eu_min) * (raw - raw_min) / (raw_max - raw_min)

for raw in [0, 8000, 16384, 24576, 32767]:
    print(f'raw={raw} → {scale(raw):.1f} °C')